## **安裝套件**

In [ ]:
# 安裝必要的 Python 庫
!pip install -U langchain langchain-core langchain-google-genai atlassian-python-api py-trello langchain-community pypdf faiss-cpu gspread oauth2client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.4/477.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.6/426.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9

In [ ]:
!pip show langchain

Name: langchain
Version: 1.1.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


## **Confluence Agent**

In [ ]:
from atlassian import Confluence
from langchain_core.tools import tool
from bs4 import BeautifulSoup
import os
from google.colab import userdata

# ✅ 從 Colab Secrets 讀取憑證
try:
    CONFLUENCE_URL = userdata.get('CONFLUENCE_URL')
    CONFLUENCE_USERNAME = userdata.get('CONFLUENCE_USERNAME')
    CONFLUENCE_API_TOKEN = userdata.get('CONFLUENCE_API_TOKEN')

    print("✅ Confluence 憑證已從 Colab Secrets 載入")
    print(f"   URL: {CONFLUENCE_URL}")
    print(f"   Username: {CONFLUENCE_USERNAME}")

except Exception as e:
    print(f"❌ 無法讀取 Colab Secrets: {e}")
    print("\n請在 Colab 左側點擊 '🔑 Secrets'，新增以下三個 secrets:")
    print("  - CONFLUENCE_URL")
    print("  - CONFLUENCE_USERNAME")
    print("  - CONFLUENCE_API_TOKEN")
    raise

# 初始化 Confluence 客戶端
confluence = Confluence(
    url=CONFLUENCE_URL,
    username=CONFLUENCE_USERNAME,
    password=CONFLUENCE_API_TOKEN,
    cloud=True
)

# 測試連線
try:
    # 使用簡單的 API 呼叫來驗證連線（獲取所有 spaces）
    spaces = confluence.get_all_spaces(limit=1)
    print(f"✅ Confluence 連線成功！")
except Exception as e:
    print(f"❌ Confluence 連線失敗: {e}")
    print("\n請檢查:")
    print("  1. CONFLUENCE_URL 格式是否正確 (例如: https://your-domain.atlassian.net)")
    print("  2. CONFLUENCE_USERNAME 是否為正確的登入 Email")
    print("  3. CONFLUENCE_API_TOKEN 是否有效 (可至 https://id.atlassian.com/manage-profile/security/api-tokens 重新生成)")
    raise


def clean_html(html_content):
    """輔助函式：將 Confluence 的 HTML 轉換為純文字，減少 Token 消耗"""
    soup = BeautifulSoup(html_content, "html.parser")
    return soup.get_text(separator="\n")


# --- 定義給 Agent 使用的工具 (Tools) ---

@tool
def search_confluence_pages(query: str) -> str:
    """
    當不知道確切頁面標題時，使用此工具搜索 Confluence 頁面。
    輸入關鍵字，返回相關頁面的 ID 和標題列表。
    """
    print(f"\n[Tool Call] 正在搜尋 Confluence: {query} ...")
    try:
        # 使用 CQL (Confluence Query Language) 進行搜索
        results = confluence.cql(f'text ~ "{query}"', limit=5)

        output = []
        for item in results.get("results", []):
            page_id = item.get("content", {}).get("id")
            title = item.get("content", {}).get("title")
            url = item.get("content", {}).get("_links", {}).get("webui")
            output.append(f"ID: {page_id} | Title: {title} | URL: {CONFLUENCE_URL}{url}")

        return "\n".join(output) if output else "找不到相關頁面。"
    except Exception as e:
        return f"搜尋錯誤: {str(e)}"


@tool
def get_confluence_page_content(page_id: str) -> str:
    """
    當已經知道 Page ID 時，使用此工具讀取該頁面的詳細內容。
    這會獲取'最新版本'的內容。
    """
    print(f"\n[Tool Call] 正在讀取頁面 ID: {page_id} 的內容 ...")
    try:
        page = confluence.get_page_by_id(page_id, expand='body.storage,version')
        title = page.get("title")
        version = page.get("version", {}).get("number")
        body_html = page.get("body", {}).get("storage", {}).get("value", "")

        # 讀取 Comments (隱性知識)
        comments_data = confluence.get_page_comments(page_id, expand='body.storage', depth="all")
        comments_text = ""
        if comments_data.get('results'):
            comments_text = "\n\n--- User Comments (隱性知識) ---\n"
            for c in comments_data['results']:
                comments_text += f"- {clean_html(c['body']['storage']['value'])}\n"

        clean_text = clean_html(body_html)

        return f"標題: {title} (Ver.{version})\n內容:\n{clean_text}\n{comments_text}"
    except Exception as e:
        return f"讀取錯誤: {str(e)}"


# 工具列表
confluence_tools = [search_confluence_pages, get_confluence_page_content]

print("\n✅ Confluence Agent 工具已就緒")
print(f"   可用工具: {[tool.name for tool in confluence_tools]}")

✅ Confluence 憑證已從 Colab Secrets 載入
   URL: https://ikea-data-team-toolbox.atlassian.net
   Username: jtung@ikea.com.tw
✅ Confluence 連線成功！

✅ Confluence Agent 工具已就緒
   可用工具: ['search_confluence_pages', 'get_confluence_page_content']


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from google.colab import userdata

# 1. 初始化 LLM (使用 Gemini model，支援強大的上下文和推理)
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0,
    max_retries=2,
    google_api_key=userdata.get('gemini_api_key')
)

# 2. 定義 Agent 的 Prompt (修改點：加入 chat_history)
confluence_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    你是一個專業的知識管理助手，你的任務是幫助用戶查詢公司 Confluence 知識庫中的「最新」資訊。

    規則：
    1. 優先使用工具查詢，不要憑空捏造。
    2. 若用戶的問題承接上文（例如：「那這個流程的負責人是誰？」），請參考對話歷史 (Chat History) 來理解上下文。
    3. 回答請簡潔並標註來源。
    4. 如果用戶要求列出「所有」頁面，請解釋你的工具無法直接列出所有頁面（因為數量龐大且可能涉及隱私），並主動引導用戶提供具體關鍵字來進行搜索。
    5. 來源標準格式為請統一使用 **`https://ikea-data-team-toolbox.atlassian.net/wiki/...`** 這樣的標準格式。
    """),
    # [重要修改] 這裡會自動填入過去的對話紀錄
    MessagesPlaceholder(variable_name="chat_history"),

    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 3. 建立 Agent
confluence_agent = create_tool_calling_agent(llm, confluence_tools, confluence_prompt)
confluence_executor = AgentExecutor(agent=confluence_agent, tools=confluence_tools, verbose=True)

print("Confluence Agent 已就緒。")

Confluence Agent 已就緒。


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

# 初始化一個空的對話歷史清單
# 如果您想重置記憶，請重新執行這一行，否則舊記憶會一直保留
if 'chat_history' not in locals():
    chat_history = []

print("--- 開始互動式對話 (輸入 'exit' 或 'quit' 離開) ---")

while True:
    # 1. 獲取用戶輸入
    user_input = input("\n你: ")

    # 檢查是否要結束
    if user_input.lower() in ["exit", "quit", "q"]:
        print("對話結束。")
        break

    if not user_input.strip():
        continue

    try:
        # 2. 呼叫 Agent，並傳入目前的對話歷史
        response = confluence_executor.invoke({
            "input": user_input,
            "chat_history": chat_history
        })

        # 修正輸出格式：從回應中提取純文字內容
        agent_response_raw = response["output"]
        agent_response_parts = []
        if isinstance(agent_response_raw, list):
            for item in agent_response_raw:
                if isinstance(item, dict) and 'text' in item:
                    agent_response_parts.append(item['text'])
                elif isinstance(item, str):
                    agent_response_parts.append(item)
            agent_response = "".join(agent_response_parts)
        else:
            agent_response = str(agent_response_raw)

        print(f"Agent: {agent_response}")

        # 3. 更新記憶 (現在這裡已經有定義 HumanMessage 了)
        chat_history.extend([
            HumanMessage(content=user_input),
            AIMessage(content=agent_response)
        ])

    except Exception as e:
        print(f"發生錯誤: {e}")

--- 開始互動式對話 (輸入 'exit' 或 'quit' 離開) ---

你: 請你解釋甚麼是ikea helpdesk


> Entering new AgentExecutor chain...

Invoking: `search_confluence_pages` with `{'query': 'ikea helpdesk'}`



[Tool Call] 正在搜尋 Confluence: ikea helpdesk ...
ID: 98589 | Title: CDP Knowledge Management | URL: https://ikea-data-team-toolbox.atlassian.net/spaces/idtt/pages/98589/CDP+Knowledge+Management
ID: att8224772 | Title: IKEA Data HelpDesk Roadmap Planning - Architecture Diagram_v1.1.jpg | URL: https://ikea-data-team-toolbox.atlassian.net/pages/viewpageattachments.action?pageId=1376517&preview=%2F1376517%2F8224772%2FIKEA+Data+HelpDesk+Roadmap+Planning+-+Architecture+Diagram_v1.1.jpg
ID: 76972034 | Title: Data Requirements Specification Template | URL: https://ikea-data-team-toolbox.atlassian.net/spaces/idtt/pages/76972034/Data+Requirements+Specification+Template
ID: 20348950 | Title: 3. System Developer Docs | URL: https://ikea-data-team-toolbox.atlassian.net/spaces/idtt/pages/20348950/3.+System+Developer+Docs
ID: 

## **Trello Agent**

In [ ]:
from trello import TrelloClient

# 直接測試（先硬編碼測試，確認沒問題後再改用環境變數）
test_client = TrelloClient(
    api_key='becaf275837aeb8c0de80ebc33ef30a5',
    token='ATTAd3894cf4ea01e3ae3a9e137a209a8dba349c7cad24f78bb5c6c45738d3c909bf15A253C9'
)

# 測試是否能讀取看板
try:
    boards = test_client.list_boards()
    print(f"成功！找到 {len(boards)} 個看板")
    for board in boards[:10]:  # 只印前 3 個
        print(f"  - {board.name}")
except Exception as e:
    print(f"失敗: {e}")

成功！找到 6 個看板
  - HK App3.0 GA
  - IKEA Data HelpDesk Project
  - IKEA Data HelpDesk Project
  - IKEA Data Requests
  - TW - GA 4 - UAT
  - TW APP3.0 GA QA


In [ ]:
from trello import TrelloClient
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from google.colab import userdata
import dateutil.parser
import os

# --- 1. 設定與環境變數 ---
TARGET_BOARD_ID = "67fccfb26a69c06a792c59b2"
TARGET_BOARD_NAME = "IKEA Data Requests"

# 從 Colab Secrets 讀取憑證
os.environ["TRELLO_API_KEY"] = userdata.get('TRELLO_API_KEY')
os.environ["TRELLO_TOKEN"] = userdata.get('TRELLO_TOKEN')

# ✅ 修正：使用 token 參數而非 api_secret
client = TrelloClient(
    api_key=os.environ["TRELLO_API_KEY"],
    token=os.environ["TRELLO_TOKEN"]
)

# 測試連線
try:
    boards = client.list_boards()
    print(f"✅ Trello 連線成功！找到 {len(boards)} 個看板")
except Exception as e:
    print(f"❌ 連線失敗: {e}")

# 3. 輔助函式：格式化日期
def format_date(date_obj):
    if not date_obj: return "無"
    try:
        if hasattr(date_obj, 'strftime'): return date_obj.strftime("%Y-%m-%d %H:%M")
        dt = dateutil.parser.parse(str(date_obj))
        return dt.strftime("%Y-%m-%d %H:%M")
    except: return str(date_obj)

# --- 4. 定義專用工具 (移除 list_boards，簡化邏輯) ---

@tool
def get_project_status() -> str: # <--- 修改點：不再需要輸入 board_id
    """
    讀取 'IKEA Data Requests' 看板中所有的清單與卡片概覽。
    當用戶詢問「目前的進度」、「有哪些任務」或「待辦事項」時使用此工具。
    """
    try:
        # 直接使用鎖定的 ID
        board = client.get_board(TARGET_BOARD_ID)
        result = f"--- 專案看板: {board.name} (ID: {TARGET_BOARD_ID}) ---\n"

        for lst in board.list_lists():
            result += f"\n[List: {lst.name}]\n"
            cards = lst.list_cards()
            if not cards: result += "  (無卡片)\n"
            for card in cards:
                labels = [l.name for l in card.labels if l.name]
                label_str = f"[{','.join(labels)}]" if labels else ""
                due_str = f"Due:{format_date(card.due)}" if card.due else ""
                # 這裡列出卡片名稱與 ID，方便 Agent 下一步查細節
                result += f"  - {card.name} (ID: {card.id}) {label_str} {due_str}\n"
        return result
    except Exception as e:
        return f"讀取看板失敗: {e}"

@tool
def get_card_details(card_id: str) -> str:
    """
    讀取特定卡片的詳細內容。
    包含 Start/End Date, Labels 與所有留言 (Comments)。
    """
    try:
        card = client.get_card(card_id)

        # 安全讀取屬性
        raw_start = getattr(card, 'start', None)
        if raw_start is None and hasattr(card, '_json'):
            raw_start = card._json.get('start')
        raw_due = getattr(card, 'due', None)
        desc = getattr(card, 'description', '') or getattr(card, 'desc', '(無描述)')
        labels = [f"{l.name}" for l in card.labels if l.name]

        details = f"=== 卡片詳情: {card.name} ===\n"
        details += f"📁 List: {client.get_list(card.list_id).name}\n"
        details += f"🏷️ Labels: {', '.join(labels) if labels else '無'}\n"
        details += f"📅 Start: {format_date(raw_start)} | Due: {format_date(raw_due)}\n"
        details += f"📝 Description:\n{desc}\n"

        # 讀取留言
        comments = card.get_comments()
        details += "\n--- Comments ---\n"
        if comments:
            for comment in comments:
                author = comment.get('memberCreator', {}).get('fullName', 'Unknown')
                text = comment.get('data', {}).get('text', '')
                date = format_date(comment.get('date', ''))
                details += f"[{date}] {author}: {text}\n"
        else:
            details += "(無留言)\n"
        return details
    except Exception as e:
        return f"讀取卡片詳情失敗: {e}"

trello_tools = [get_project_status, get_card_details]

print("Trello Agent 工具已更新。")


✅ Trello 連線成功！找到 6 個看板
Trello Agent 工具已更新。


In [ ]:
print(f"API Key Load Status: {bool(os.environ.get('TRELLO_API_KEY'))}")
print(f"Token Load Status: {bool(os.environ.get('TRELLO_TOKEN'))}")

API Key Load Status: True
Token Load Status: True


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from google.colab import userdata

# 1. 初始化 LLM (修正模型名稱)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_retries=2,
    google_api_key=userdata.get('gemini_api_key')
)

# 2. 定義 Trello Agent 的 Prompt
trello_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    你是一位專業的專案管理助手，目前你只負責管理 "IKEA Data Requests" 這個專案。

    你的工具與查詢策略如下：
    1. **查詢概況**：當用戶問「有哪些任務」、「進度如何」或「待辦事項」時，請直接呼叫 `get_project_status` (不需參數)。這會列出所有卡片與標籤。
    2. **查詢細節**：當用戶問特定任務（例如 "Search dashboard"）的細節或留言時，請先從概況中找到該卡片的 ID，然後呼叫 `get_card_details`。

    關於時間與標籤的邏輯：
    - **時程問題**：關注 `Start Date` 和 `End Date`。若過期請提醒用戶。
    - **分類問題**：利用 `Labels` 資訊來篩選 (如 Bug, Data, Enhancement)。
    - **進度問題**：檢查 `Completed` 狀態。

    重要規則：
    - 若發現卡片中有重要的留言討論（如變更需求、Bug原因），請務必總結出來。
    - 請直接回答與 "IKEA Data Requests" 相關的內容，不要嘗試搜尋其他看板。
    """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 3. 建立 Agent
# 注意：這裡的 trello_tools 必須是上一一個步驟定義的 (包含 get_project_status 那組)
trello_agent = create_tool_calling_agent(llm, trello_tools, trello_prompt)
trello_executor = AgentExecutor(agent=trello_agent, tools=trello_tools, verbose=True)

print("Trello Agent 就緒。")

Trello Agent 就緒。


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

# 初始化記憶
trello_history = []

print("--- Trello Agent 對話開始 (輸入 'q' 離開) ---")

while True:
    user_input = input("\nUser: ")
    if user_input.lower() in ['q', 'exit']:
        break

    try:
        response = trello_executor.invoke({
            "input": user_input,
            "chat_history": trello_history
        })

        agent_response_raw = response["output"]
        agent_response_parts = []
        if isinstance(agent_response_raw, list):
            for item in agent_response_raw:
                if isinstance(item, dict) and 'text' in item:
                    agent_response_parts.append(item['text'])
                elif isinstance(item, str):
                    agent_response_parts.append(item)
            agent_response = "".join(agent_response_parts)
        else:
            agent_response = str(agent_response_raw)

        print(f"Agent: {agent_response}")

        # 更新記憶
        trello_history.extend([
            HumanMessage(content=user_input),
            AIMessage(content=agent_response_raw)
        ])

    except Exception as e:
        print(f"Error: {e}")

--- Trello Agent 對話開始 (輸入 'q' 離開) ---

User: 請整理正在進行專案的留言內容債要


> Entering new AgentExecutor chain...

Invoking: `get_project_status` with `{}`


--- 專案看板: IKEA Data Requests (ID: 67fccfb26a69c06a792c59b2) ---

[List: Read the Guidelines]
  (無卡片)

[List: [READ] How to add the card?]
  - Step 1:  建立卡片與需求 (ID: 68dc892a4220e1e2258e9e85)  
  - Step 2: 設定關鍵資訊 (ID: 68dc8a5705e8a144ef179517)  
  - Step 3: 進度追蹤與溝通 (ID: 68dc8a84fd99f1fe0ec5b9f1)  
  - Step 4: 狀態管理 (ID: 68dc8a9a69d636c845c6bb26)  
  - Step 5: 結案處理 (ID: 68dc8aaea8cd371b02d9d467)  

[List: To Do]
  - HK unique customer data request (ID: 69314fd9f34257c817b95041) [HK,Data report,CDP] Due:2025-12-17 16:00

[List: Doing]
  - PMA dashboard development (ID: 68db435207d5fb6772cb2da6) [IKNA,Dashboard,CDP] Due:2026-01-30 16:00
  - MKT headbanner optimization (NA) (ID: 68e4af1191c16d2385c04707) [IKNA,GA4,Web,Data support] 
  - Search dashboard enhancement (ID: 68db432371fbfe07137c22c9) [Dashboard,IKNA,GA4,Google Search Console] Due:2025-11

## **Document Agent**

In [ ]:
import time
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from google.colab import userdata

# 1. 設定 Embedding
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=userdata.get('gemini_api_key')
)

# 全域變數
vector_db = None

def build_document_base_robust():
    global vector_db

    # --- 2. 自動檢查檔案 (不用一直重新上傳) ---
    # 尋找當前目錄下最新的 PDF
    pdf_files = [f for f in os.listdir('.') if f.lower().endswith('.pdf')]
    if not pdf_files:
        print("❌ 找不到 PDF 檔案，請先上傳。")
        from google.colab import files
        uploaded = files.upload()
        if not uploaded: return
        filename = list(uploaded.keys())[0]
    else:
        # 自動選擇最後修改的 PDF (通常是剛剛上傳的 Guidebook (2).pdf)
        pdf_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        filename = pdf_files[0]
        print(f"📂 偵測到已上傳檔案，直接使用: {filename}")

    # 3. 讀取與切割
    print(f"正在讀取與切割 {filename} ...")
    loader = PyPDFLoader(filename)
    pages = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    splits = text_splitter.split_documents(pages)
    total_chunks = len(splits)
    print(f"文件已切割為 {total_chunks} 個區塊。準備開始 Embedding...")

    # --- 4. 具備「重試機制」的批次處理 ---
    batch_size = 5  # 降到極低：每次 5 個
    normal_sleep = 5 # 正常休息 5 秒
    error_sleep = 60 # 遇到錯誤休息 60 秒

    vector_db = None # 重置

    i = 0
    while i < total_chunks:
        batch = splits[i : i + batch_size]
        current_end = min(i + batch_size, total_chunks)
        print(f"🔄 處理進度: {i+1} ~ {current_end} / {total_chunks} ...", end=" ")

        try:
            if vector_db is None:
                vector_db = FAISS.from_documents(batch, embeddings)
            else:
                vector_db.add_documents(batch)

            print("✅ OK")
            # 成功後，推進進度
            i += batch_size
            # 正常休息，讓 API 喘口氣
            time.sleep(normal_sleep)

        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "ResourceExhausted" in error_msg:
                print(f"\n⚠️ 觸發 API 速率限制 (Rate Limit)。")
                print(f"⏳ 系統將自動暫停 {error_sleep} 秒後重試，請耐心等待...")
                time.sleep(error_sleep)
                # 這裡不執行 i += batch_size，所以迴圈會再次嘗試同一批
            else:
                print(f"\n❌ 發生未預期的錯誤: {e}")
                # 如果不是 Rate Limit，可能要停止以免死循環
                break

    print(f"\n🎉 恭喜！全部 {total_chunks} 個區塊已成功建立索引。")

# 執行
build_document_base_robust()

❌ 找不到 PDF 檔案，請先上傳。


Saving Guidebook.pdf to Guidebook.pdf
正在讀取與切割 Guidebook.pdf ...
文件已切割為 141 個區塊。準備開始 Embedding...
🔄 處理進度: 1 ~ 5 / 141 ... ✅ OK
🔄 處理進度: 6 ~ 10 / 141 ... ✅ OK
🔄 處理進度: 11 ~ 15 / 141 ... ✅ OK
🔄 處理進度: 16 ~ 20 / 141 ... ✅ OK
🔄 處理進度: 21 ~ 25 / 141 ... ✅ OK
🔄 處理進度: 26 ~ 30 / 141 ... ✅ OK
🔄 處理進度: 31 ~ 35 / 141 ... ✅ OK
🔄 處理進度: 36 ~ 40 / 141 ... ✅ OK
🔄 處理進度: 41 ~ 45 / 141 ... ✅ OK
🔄 處理進度: 46 ~ 50 / 141 ... ✅ OK
🔄 處理進度: 51 ~ 55 / 141 ... ✅ OK
🔄 處理進度: 56 ~ 60 / 141 ... ✅ OK
🔄 處理進度: 61 ~ 65 / 141 ... ✅ OK
🔄 處理進度: 66 ~ 70 / 141 ... ✅ OK
🔄 處理進度: 71 ~ 75 / 141 ... ✅ OK
🔄 處理進度: 76 ~ 80 / 141 ... ✅ OK
🔄 處理進度: 81 ~ 85 / 141 ... ✅ OK
🔄 處理進度: 86 ~ 90 / 141 ... ✅ OK
🔄 處理進度: 91 ~ 95 / 141 ... ✅ OK
🔄 處理進度: 96 ~ 100 / 141 ... ✅ OK
🔄 處理進度: 101 ~ 105 / 141 ... ✅ OK
🔄 處理進度: 106 ~ 110 / 141 ... ✅ OK
🔄 處理進度: 111 ~ 115 / 141 ... ✅ OK
🔄 處理進度: 116 ~ 120 / 141 ... ✅ OK
🔄 處理進度: 121 ~ 125 / 141 ... ✅ OK
🔄 處理進度: 126 ~ 130 / 141 ... ✅ OK
🔄 處理進度: 131 ~ 135 / 141 ... ✅ OK
🔄 處理進度: 136 ~ 140 / 141 ... ✅ OK
🔄 處理進度: 141 ~ 141 / 1

In [ ]:
from langchain_core.tools import tool

@tool
def search_document_base(query: str) -> str:
    """
    檢索 PDF 知識庫中的相關內容。
    當用戶詢問關於「規範」、「流程」、「合約細節」或「文件內容」時，必須使用此工具。
    回傳最相關的 5 個段落。
    """
    global vector_db
    if vector_db is None:
        return "錯誤：知識庫尚未建立，請先上傳 PDF 文件。"

    print(f"\n[Knowledge Search] 正在檢索: {query} ...")

    # 進行相似度搜尋 (k=5 代表回傳前五名最相關的片段)
    results = vector_db.similarity_search(query, k=5)

    output = ""
    for i, doc in enumerate(results):
        # 包含頁碼資訊，方便溯源
        page_num = doc.metadata.get("page", "未知")
        output += f"\n--- 參考片段 {i+1} (Page {page_num}) ---\n"
        output += doc.page_content
        output += "\n"

    return output

# 打包工具
document_tools = [search_document_base]
print("Document Agent 工具已更新。")

Knowledge Agent 工具已更新。


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. 初始化 LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=userdata.get('gemini_api_key')
)

# 2. 定義 Prompt
document_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    你是一位專業的文件知識專家，你的任務是根據內部的 PDF 文件回答用戶的詢問。

    回答規則：
    1. **必須使用工具**：請務必使用 `search_knowledge_base` 檢索內容，嚴禁憑空捏造。
    2. **標註來源**：在回答時，請儘量註明資訊來自文件的「第幾頁」(Page X)。
    3. **誠實回答**：如果檢索結果中沒有相關資訊，請直接說「文件中未提及此內容」，不要強行解釋。
    4. **整合資訊**：如果檢索到多個片段，請將其整合成通順的答案。
    """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 3. 建立 Agent
document_agent = create_tool_calling_agent(llm, document_tools, document_prompt)
document_executor = AgentExecutor(agent=document_agent, tools=document_tools, verbose=True)

print("Document Agent 就緒。")

Knowledge Agent 就緒。


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

# 初始化記憶
if 'document_history' not in locals():
    document_history = []

print("--- Document Agent 對話開始 (輸入 'q' 離開) ---")

while True:

    user_input = input("\nUser: ")
    if user_input.lower() in ['q', 'exit']:
        break

    if not user_input.strip(): continue

    try:
        response = document_executor.invoke({
            "input": user_input,
            "chat_history": document_history
        })

        print(f"Agent: {response['output']}")

        # 更新記憶
        document_history.extend([
            HumanMessage(content=user_input),
            AIMessage(content=response['output'])
        ])

    except Exception as e:
        print(f"Error: {e}")


--- Knowledge Agent 對話開始 (輸入 'q' 離開) ---

User: 請說明一下ikea moment of interaction


> Entering new AgentExecutor chain...
文件中並未明確提及「IKEA Moment of Interaction (MOI)」。

然而，文件中提到了「**顧客接觸的關鍵時刻 (MOT)**」(Moments of Truth) (Page 70)，這指的是顧客與品牌互動中，那些對顧客體驗和品牌認知產生重大影響的重要時間點。

雖然沒有直接定義「Moment of Interaction」，但我們可以從「顧客體驗管理 (CEM)」的五個階段中理解顧客與 IKEA 互動的不同時刻 (Page 76)：

*   **認知 (Awareness)**：顧客首次接觸 IKEA 品牌。
*   **興趣 (Interest)**：顧客主動瀏覽、探索 IKEA 的產品或服務。
*   **可望 (Desire / Consideration)**：顧客在購買前比較 IKEA 的產品，進行研究。
*   **行動 (Action / Conversion)**：顧客實際購買 IKEA 產品、使用服務或尋求售後支援。
*   **忠誠 (Loyalty)**：顧客再次購買、推薦 IKEA 產品或提供回饋。

這些 CEM 階段描述了顧客在不同時間點與 IKEA 進行的各種互動。

> Finished chain.
Agent: 文件中並未明確提及「IKEA Moment of Interaction (MOI)」。

然而，文件中提到了「**顧客接觸的關鍵時刻 (MOT)**」(Moments of Truth) (Page 70)，這指的是顧客與品牌互動中，那些對顧客體驗和品牌認知產生重大影響的重要時間點。

雖然沒有直接定義「Moment of Interaction」，但我們可以從「顧客體驗管理 (CEM)」的五個階段中理解顧客與 IKEA 互動的不同時刻 (Page 76)：

*   **認知 (Awareness)**：顧客首次接觸 IKEA 品牌。
*   **興趣 (Interest)**：顧客主動瀏覽、探索 IKEA 的產品或服務。
*   **可望 (Desir

## **Data Analyst Agent**

In [ ]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from google.colab import userdata
import gspread
import json
from oauth2client.service_account import ServiceAccountCredentials
from google.colab import drive
import re
import pandas as pd

drive.mount('/content/drive')

# 全局變量來緩存工作表數據
_cached_data = {}

@tool
def list_worksheets() -> str:
    """
    列出 Google Sheet 中所有可用的工作表名稱。
    當用戶想知道有哪些工作表可以查詢時使用此工具。
    """
    try:
        scope = ['https://spreadsheets.google.com/feeds',
                 'https://www.googleapis.com/auth/drive']
        creds = ServiceAccountCredentials.from_json_keyfile_name(
            '/content/drive/MyDrive/Colab Notebooks/cedar-unison-374003-1a6afa09bbb7.json',
            scopes=scope
        )
        gc = gspread.authorize(creds)
        spreadsheet_key = '1bnqghULmnxgZdu4ALDZ2FGUzBxwamD27qYZVGMq1uEo'
        spreadsheet = gc.open_by_key(spreadsheet_key)
        worksheet_titles = [ws.title for ws in spreadsheet.worksheets()]
        return f"此 Google Sheet 中可用的工作表有：{', '.join(worksheet_titles)}"
    except Exception as e:
        return f"列出工作表時發生錯誤: {str(e)}"


@tool
def get_worksheet_structure(worksheet_name: str) -> str:
    """
    獲取指定工作表的結構信息，包括欄位名稱、資料行數等基本信息。
    當用戶想了解工作表的架構或有哪些欄位時使用此工具。

    參數:
    - worksheet_name: 工作表的名稱
    """
    try:
        scope = ['https://spreadsheets.google.com/feeds',
                 'https://www.googleapis.com/auth/drive']
        creds = ServiceAccountCredentials.from_json_keyfile_name(
            '/content/drive/MyDrive/Colab Notebooks/cedar-unison-374003-1a6afa09bbb7.json',
            scopes=scope
        )
        gc = gspread.authorize(creds)
        spreadsheet_key = '1bnqghULmnxgZdu4ALDZ2FGUzBxwamD27qYZVGMq1uEo'
        spreadsheet = gc.open_by_key(spreadsheet_key)
        worksheet = spreadsheet.worksheet(worksheet_name)

        all_records = worksheet.get_all_records()
        _cached_data[worksheet_name] = all_records  # 緩存數據

        if not all_records:
            return f"工作表 '{worksheet_name}' 是空的，沒有資料。"

        # 獲取欄位名稱
        columns = list(all_records[0].keys())
        row_count = len(all_records)

        # 顯示每個欄位的一些統計信息
        result = f"工作表 '{worksheet_name}' 的結構信息：\n"
        result += f"- 總資料行數: {row_count}\n"
        result += f"- 欄位列表 ({len(columns)} 個): {', '.join(columns)}\n"
        result += f"\n前 3 筆資料範例：\n"

        for i, record in enumerate(all_records[:3], 1):
            result += f"\n第 {i} 筆:\n"
            for key, value in record.items():
                result += f"  - {key}: {value}\n"

        return result
    except gspread.exceptions.WorksheetNotFound:
        return f"找不到名為 '{worksheet_name}' 的工作表。請使用 list_worksheets 工具查看可用的工作表。"
    except Exception as e:
        return f"讀取工作表結構時發生錯誤: {str(e)}"


@tool
def query_worksheet_data(worksheet_name: str, query_description: str) -> str:
    """
    根據查詢需求從指定工作表中檢索和分析資料。
    此工具可以回答關於資料的各種問題，例如：
    - 統計數量（有多少筆資料、某狀態有幾筆等）
    - 篩選資料（找出符合特定條件的記錄）
    - 摘要信息（列出所有類別、統計各狀態數量等）

    參數:
    - worksheet_name: 工作表的名稱
    - query_description: 詳細描述要查詢的內容，例如 "統計各狀態的工單數量"、"列出所有待處理的工單"
    """
    try:
        # 先檢查緩存
        if worksheet_name not in _cached_data:
            scope = ['https://spreadsheets.google.com/feeds',
                     'https://www.googleapis.com/auth/drive']
            creds = ServiceAccountCredentials.from_json_keyfile_name(
                '/content/drive/MyDrive/Colab Notebooks/cedar-unison-374003-1a6afa09bbb7.json',
                scopes=scope
            )
            gc = gspread.authorize(creds)
            spreadsheet_key = '1bnqghULmnxgZdu4ALDZ2FGUzBxwamD27qYZVGMq1uEo'
            spreadsheet = gc.open_by_key(spreadsheet_key)
            worksheet = spreadsheet.worksheet(worksheet_name)
            all_records = worksheet.get_all_records()
            _cached_data[worksheet_name] = all_records
        else:
            all_records = _cached_data[worksheet_name]

        if not all_records:
            return f"工作表 '{worksheet_name}' 中沒有資料。"

        # 將資料轉換為 DataFrame 以便於分析
        df = pd.DataFrame(all_records)

        result = f"針對工作表 '{worksheet_name}' 的查詢結果：\n\n"
        result += f"總資料筆數: {len(df)}\n\n"

        # 根據查詢描述提供相應的分析
        query_lower = query_description.lower()

        # 提供資料摘要
        result += "=== 資料摘要 ===\n"
        result += f"欄位: {', '.join(df.columns.tolist())}\n\n"

        # 對每個欄位進行統計
        for col in df.columns:
            if df[col].dtype == 'object' or df[col].dtype == 'string':
                unique_values = df[col].nunique()
                if unique_values <= 20:  # 只顯示類別不太多的欄位統計
                    result += f"【{col}】的分布:\n"
                    value_counts = df[col].value_counts()
                    for value, count in value_counts.items():
                        result += f"  - {value}: {count} 筆\n"
                    result += "\n"

        # 顯示所有資料（如果不多的話）或最近的資料
        if len(df) <= 10:
            result += "=== 所有資料 ===\n"
            for idx, row in df.iterrows():
                result += f"\n第 {idx + 1} 筆:\n"
                for col in df.columns:
                    result += f"  - {col}: {row[col]}\n"
        else:
            result += f"=== 最近 10 筆資料 ===\n"
            for idx in range(min(10, len(df))):
                row = df.iloc[idx]
                result += f"\n第 {idx + 1} 筆:\n"
                for col in df.columns:
                    result += f"  - {col}: {row[col]}\n"
            result += f"\n(僅顯示前 10 筆，共 {len(df)} 筆資料)"

        return result

    except gspread.exceptions.WorksheetNotFound:
        return f"找不到名為 '{worksheet_name}' 的工作表。"
    except Exception as e:
        return f"查詢資料時發生錯誤: {str(e)}"


# Data Analyst Agent 的工具列表
analyst_tools = [list_worksheets, get_worksheet_structure, query_worksheet_data]

# 初始化 LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=userdata.get('gemini_api_key')
)

# 定義 Data Analyst Agent 的 Prompt
analyst_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    你是一位專門負責分析和回答IKEA Data HelpDesk資料相關問題的助手。

    你有三個工具可以使用：
    1. list_worksheets: 列出所有可用的工作表
    2. get_worksheet_structure: 獲取工作表的結構和欄位信息
    3. query_worksheet_data: 查詢和分析工作表中的資料

    工作流程建議：
    - 如果用戶不確定要查詢哪個工作表，先使用 list_worksheets
    - 如果用戶想了解工作表有哪些欄位，使用 get_worksheet_structure
    - 如果用戶想查詢具體資料或統計資訊，使用 query_worksheet_data

    回答時請：
    - 使用繁體中文
    - 提供清晰的資料摘要和分析
    - 如果資料量大，提供關鍵統計和重點資訊
    - 主動提出可以進一步查詢的方向
    """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 建立 Agent
analyst_agent = create_tool_calling_agent(llm, analyst_tools, analyst_prompt)
analyst_executor = AgentExecutor(agent=analyst_agent, tools=analyst_tools, verbose=True)

print("Analyst Agent 已就緒！")
print("\n功能說明：")
print("1. 列出所有工作表")
print("2. 查看工作表結構和欄位")
print("3. 查詢和分析資料（包含統計、摘要、篩選等）")
print("\n範例使用：")
print("- '列出所有工作表'")
print("- '顯示 Request 工作表的結構'")
print("- '統計 Request 工作表中各狀態的數量'")
print("- '顯示 Request 工作表的所有資料'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Analyst Agent 已就緒！

功能說明：
1. 列出所有工作表
2. 查看工作表結構和欄位
3. 查詢和分析資料（包含統計、摘要、篩選等）

範例使用：
- '列出所有工作表'
- '顯示 Request 工作表的結構'
- '統計 Request 工作表中各狀態的數量'
- '顯示 Request 工作表的所有資料'


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

# 初始化記憶
if 'analyst_history' not in locals():
    analyst_history = []

response = analyst_executor.invoke({
    "input": "請列出所有工作表",
    "chat_history": analyst_history
})
print(f"Agent: {response['output']}")
analyst_history.extend([
    HumanMessage(content="請列出所有工作表"),
    AIMessage(content=response['output'])
])



> Entering new AgentExecutor chain...

Invoking: `list_worksheets` with `{}`


此 Google Sheet 中可用的工作表有：Request, Feedback, Metrics List, KnowledgeBaseIKEA Data HelpDesk 中有以下工作表可供查詢：Request, Feedback, Metrics List, KnowledgeBase。

請問您對哪個工作表感興趣，或者有什麼特定的問題想查詢嗎？

> Finished chain.
Agent: IKEA Data HelpDesk 中有以下工作表可供查詢：Request, Feedback, Metrics List, KnowledgeBase。

請問您對哪個工作表感興趣，或者有什麼特定的問題想查詢嗎？


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

# 初始化記憶
if 'analyst_history' not in locals():
    analyst_history = []

print("--- Document Agent 對話開始 (輸入 'q' 離開) ---")

while True:

    user_input = input("\nUser: ")
    if user_input.lower() in ['q', 'exit']:
        break

    if not user_input.strip(): continue

    try:
        response = analyst_executor.invoke({
            "input": user_input,
            "chat_history": analyst_history
        })

        agent_response_raw = response["output"]
        agent_response_parts = []
        if isinstance(agent_response_raw, list):
            for item in agent_response_raw:
                if isinstance(item, dict) and 'text' in item:
                    agent_response_parts.append(item['text'])
                elif isinstance(item, str):
                    agent_response_parts.append(item)
            agent_response = "".join(agent_response_parts)
        else:
            agent_response = str(agent_response_raw)

        print(f"Agent: {agent_response}")

        # 更新記憶
        analyst_history.extend([
            HumanMessage(content=user_input),
            AIMessage(content=response['output'])
        ])

    except Exception as e:
        print(f"Error: {e}")


--- Document Agent 對話開始 (輸入 'q' 離開) ---

User: 你是誰


> Entering new AgentExecutor chain...
[{'type': 'text', 'text': '我是一個專為IKEA Data HelpDesk資料相關問題設計的AI助手。我可以幫助您查詢、分析和回答關於這些資料的問題。', 'extras': {'signature': 'CikBcsjafHYqellzTnXBOm3a4cVPjlcS4lNoN3w5M4UFWR6ItXJ6KKsLJwp2AXLI2nxFrk9TZYtyoFFcw55ePDg8THlX5og94bDSQ3qvWop7PnUDGWw5d2YxnhmWOHuj9yxoEUImrFaKreB9qUXmxFvox/vyU9g9oRKZI5gMY3hQe0dv9fdm+FH39J2Vp8Cx4OsBgnXZB4xKJ42xMdnjpMPN4gqiAQFyyNp8x77k0MTXeIcMWgj4MMQlXMHSP7fLECmUJN9F/LzBFrrdZJJoX9k22Ru2jsnOFRlefeL5LJAXythnR54UsMevr1FAcT7EgpJmMv+QZ0Eop3NdaizDmZyGM42rbLsrEzfstIwvsSyY2+7pbnMIeztyt1mKIr1O3BbTH7FE/qnoAvUSVZwecRL3bzqvuBJa3nhtZzaE4Yhm/mPVYPv+eA=='}, 'index': 0}]

> Finished chain.
Agent: 我是一個專為IKEA Data HelpDesk資料相關問題設計的AI助手。我可以幫助您查詢、分析和回答關於這些資料的問題。

User: 請計算目前有幾張ticket


> Entering new AgentExecutor chain...

Invoking: `query_worksheet_data` with `{'worksheet_name': 'Request', 'query_description': '計算所有工單的數量'}`


針對工作表 'Request' 的查詢結果：

總資料筆數: 20

=== 資料摘要 ===
欄位: ID, Ticket No., Creation 

## **Coordinator Agent**

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# --- 1. 封裝 Trello Agent ---
@tool
def ask_trello_agent(query: str) -> str:
    """
    【專案進度專家】
    當用戶詢問關於「IKEA Data Requests」專案的進度、卡片狀態、截止日期、標籤或留言細節時，
    請呼叫此工具。
    """
    print(f"\n[Coordinator] 正在轉交給 Trello Agent: {query}")
    try:
        # 呼叫之前建立的 executor
        # 注意：這裡我們傳入一個空的 chat_history，因為子 Agent 不需要知道太多上層的廢話
        response = trello_executor.invoke({"input": query, "chat_history": []})
        return response["output"]
    except Exception as e:
        return f"Trello Agent 發生錯誤: {e}"

# --- 2. 封裝 Document Agent (PDF) ---
@tool
def ask_document_agent(query: str) -> str:
    """
    【交接文件專家】
    當用戶詢問關於「交接手冊內容」內的知識時，
    請呼叫此工具。
    """
    print(f"\n[Coordinator] 正在轉交給 Document Agent: {query}")
    try:
        response = document_executor.invoke({"input": query, "chat_history": []})
        return response["output"]
    except NameError:
        return "錯誤：Document Agent 尚未初始化，請確認是否已執行 PDF 索引步驟。"
    except Exception as e:
        return f"Document Agent 發生錯誤: {e}"

# --- 3. 封裝 Confluence Agent ---
@tool
def ask_confluence_agent(query: str) -> str:
    """
    【Data team toolbox專家】
    當用戶詢問關於「資料需求流程」、「SOP」、「建置dasboard」或需要查詢 Confluence 上的最新文件時，
    請呼叫此工具。
    """
    print(f"\n[Coordinator] 正在轉交給 Confluence Agent: {query}")
    try:

        if 'agent_executor' in globals():
            target_executor = confluence_executor
        elif 'confluence_executor' in globals():
            target_executor = confluence_executor
        else:
            return "錯誤：Confluence Agent 尚未初始化。"

        response = target_executor.invoke({"input": query, "chat_history": []})
        return response["output"]
    except Exception as e:
        return f"Confluence Agent 發生錯誤: {e}"

# --- 4. 封裝 Analyst Agent ---
@tool
def ask_analyst_agent(query: str) -> str:
    """
    【Data Analyst專家】
    當用戶詢問關於「統計Ticket數量」、「統計Ticket尚未closed的數量」的查詢，
    請呼叫此工具。
    """
    print(f"\n[Coordinator] 正在轉交給 Analyst Agent: {query}")
    try:

        if 'agent_executor' in globals():
            target_executor = analyst_executor
        elif 'confluence_executor' in globals():
            target_executor = analyst_executor
        else:
            return "錯誤：Data Analyst Agent 尚未初始化。"

        response = target_executor.invoke({"input": query, "chat_history": []})
        return response["output"]
    except Exception as e:
        return f"錯誤：Analyst Agent 發生錯誤: {e}"

# --- 4. 打包給主管的工具箱 ---
coordinator_tools = [ask_trello_agent, ask_document_agent, ask_confluence_agent, ask_analyst_agent]

print("✅ 四位專家已封裝完畢，準備指派給協調者。")

✅ 四位專家已封裝完畢，準備指派給協調者。


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from google.colab import userdata

# 1. 初始化主管級 LLM
llm_coordinator = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    google_api_key=userdata.get('gemini_api_key')
)

# 2. 定義主管的 Prompt
coordinator_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    # Role & Persona
    你是一個多模態系統的**資深專案協調官 (Senior Orchestrator)**。
    你的職責不是單純的轉接電話，而是**理解業務脈絡**，將使用者的模糊需求拆解為具體的執行計畫，並指揮身後的四位專家代理人協作。
    請保持「專業、俐落且樂於協助」的同事風格，回覆時請歸納重點，避免機械式的覆述。

    # Expert Capabilities (專家能力矩陣)
    請根據以下分工邏輯進行調度（可單選或多選）：

    1. **Trello Agent (`ask_trello_agent`)** - [專案執行現況]
       - **核心職責**：查詢「正在發生」的任務狀態。
       - **適用場景**：專案進度追蹤、卡片截止日、誰負責什麼任務、Bug 修復進度、IKEA 看板上的具體留言。
       - **決策依據**：如果問題涉及「進度」、「Deadline」、「誰在做」、「還剩多少工作」，找它。

    2. **Document Agent (`ask_document_agent`)** - [靜態規範與合約]
       - **核心職責**：查詢「既定事實」的文件與規範。
       - **適用場景**：PDF 格式的交接手冊、正式合約條款、產品規格書(Spec)、硬性規定的 SOP。
       - **決策依據**：如果問題涉及「規範是什麼」、「合約怎麼寫」、「原始規格」，找它。

    3. **Confluence Agent (`ask_confluence_agent`)** - [團隊知識與流程]
       - **核心職責**：查詢「團隊內部的 Know-How」。
       - **適用場景**：數據團隊的知識庫、操作手冊(How-to)、CDP/Dynamic Yield 的設定教學、IKEA Data Helpdesk 流程。
       - **決策依據**：如果問題涉及「怎麼操作」、「團隊筆記」、「某個工具的說明」，找它。

    4. **Data Analyst Agent (`ask_da_agent`)** - [量化數據統計]
       - **核心職責**：查詢「統計數字與儀表板」。
       - **適用場景**：Ticket 工單數量統計、儀表板(Dashboard)上的 KPI、工單處理效率、量化趨勢。
       - **決策依據**：如果問題涉及「多少(How many)」、「數據統計」、「趨勢分析」，找它。

    # Workflow (思考與決策流程)
    當收到用戶請求時，請遵循以下步驟：
    1. **分析意圖**：用戶問的是「數據」、「進度」還是「規範」？
    2. **任務拆解**：如果是複合問題（例如：「查一下目前落後的任務(Trello)，並調出相關規範(Document)」），請**同時**呼叫多個工具。
    3. **工具調用**：準確選擇對應的 Agent，不要猶豫。
    4. **資訊整合**：收到工具回傳後，請將破碎的資訊整合成一段通順的摘要回覆用戶。

    # Constraints (行為準則)
    - **禁止幻覺**：如果專家回傳「查無資料」，請如實告知用戶，不可編造數據或進度。
    - **打招呼處理**：若用戶僅輸入 "Hi", "你好"，請直接以友善態度回應，無需呼叫工具。
    - **釐清歧義**：如果 `Document` (規範) 與 `Confluence` (操作流程) 難以區分，優先思考資料來源是「正式文件」還是「網頁筆記」。
    """)
])

# 3. 建立協調者 Agent
coordinator_agent = create_tool_calling_agent(llm_coordinator, coordinator_tools, coordinator_prompt)
coordinator_executor = AgentExecutor(agent=coordinator_agent, tools=coordinator_tools, verbose=True)

print("🚀 總協調官 (Coordinator) 已就緒！請開始提問。")

🚀 總協調官 (Coordinator) 已就緒！請開始提問。


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

# 初始化全域記憶
if 'coordinator_history' not in locals():
    coordinator_history = []

print("--- 智慧交接系統 (輸入 'q' 離開) ---")
print("您可以問：")
print("1. (Trello) 目前 IKEA 專案有哪些緊急任務？")
print("2. (Document) 交接手冊裡關於App資料是什麼？")
print("3. (Confluence) 最新的CDP知識內容？")
print("4. (Analyst) 統計 Request 工作表中各狀態的數量？")
print("---------------------------------------")

while True:
    user_input = input("\nUser: ")
    if user_input.lower() in ['q', 'exit']:
        break

    if not user_input.strip(): continue

    try:
        # 呼叫協調者
        response = coordinator_executor.invoke({
            "input": user_input,
            "chat_history": coordinator_history
        })

        agent_response_raw = response["output"]
        agent_response_parts = []
        if isinstance(agent_response_raw, list):
            for item in agent_response_raw:
                if isinstance(item, dict) and 'text' in item:
                    agent_response_parts.append(item['text'])
                elif isinstance(item, str):
                    agent_response_parts.append(item)
            agent_response = "".join(agent_response_parts)
        else:
            agent_response = str(agent_response_raw)

        print(f"Agent: {agent_response}")

        # 更新記憶
        coordinator_history.extend([
            HumanMessage(content=user_input),
            AIMessage(content=response['output'])
        ])

    except Exception as e:
        print(f"系統錯誤: {e}")

--- 智慧交接系統 (輸入 'q' 離開) ---
您可以問：
1. (Trello) 目前 IKEA 專案有哪些緊急任務？
2. (Knowledge) 交接手冊裡關於App資料是什麼？
3. (Confluence) 最新的CDP知識內容？
4. (Dashboard) 統計 Request 工作表中各狀態的數量？
---------------------------------------

User: 請統計目前有多少張ticket


> Entering new AgentExecutor chain...

Invoking: `ask_dashboard_agent` with `{'query': '請統計目前有多少張ticket'}`
responded: 好的，這就請 Dashboard Agent 幫您統計目前有多少張 Ticket。



[Coordinator] 正在轉交給 Dashboard Agent: 請統計目前有多少張ticket


> Entering new AgentExecutor chain...
[{'type': 'text', 'text': '您想統計的是哪一個工作表的 ticket 數量呢？請先告訴我工作表的名稱，或者我可以先列出所有可用的工作表給您參考。', 'extras': {'signature': 'CiMBcsjafKco0xih9ZvamqdOP8xZSvGv6GX4AXAk5FfG4PPAcwpnAXLI2nx4bAYMhGMa2RCyyVU1GLBQxacI+6qj/1g8YJ+Y87Hb80p92KalDA2qtSXJo875WZa9BQDP9lSkHPwAXrDrU1zguYo2WSSRz8LQHItMqf7AyB4o64AyW2Bd/gNOl1ZhVhuOAgrgAQFyyNp8RbKK+QPtNyaepqpPgkurypHBZO4zD9BLCq2XZXfYWNTqi51lf6JmmygGTIYx+/qYt60x+No7udIpJthKBkIRUHIDvH3SsSnCPGLfWR18zevJtpE4S9kSHm3gi4m9iN7ISQ0uAMhIrtsvGo4j7NDcsZa58dzCal29WS+igcB8ZTU2dWZHlwaAFVo6DuhLsgUijb0FqVh0+SdK